# 36 - QUAL-002 / QUAL-004: Held-out con Ground Truth y evaluación Dice/IoU

Objetivo:

1. Buscar si ya existe un split/inventario para no inventar un test.
2. Armar carpetas `test-dir/images/` y `test-dir/masks/` para sagital y axial.
3. Ejecutar `scripts/evaluate_model.py` contra los checkpoints finales.
4. Guardar:
   - `docs/qual-004-sagittal.json`
   - `docs/qual-004-axial.json`
   - logs de ejecución
   - manifest del held-out usado
   - resumen para pegarle a Claude/Codex

Importante:
- Si no se confirma que los pacientes/casos nunca estuvieron en entrenamiento, el resultado debe etiquetarse como **evaluación preliminar con ground truth**, no como test limpio final.
- Esto no representa validación clínica.

In [2]:
# ============================================
# 0) Montar Google Drive e instalar dependencias
# ============================================
from google.colab import drive
drive.mount('/content/drive')

!pip -q install SimpleITK pydicom nibabel pandas pillow

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 50.3 MB/s eta 0:00:00


In [3]:
# ============================================
# 1) Configuración principal
# ============================================
from pathlib import Path
import os, sys, json, shutil, subprocess, textwrap, hashlib, csv, re
from datetime import datetime, timezone

import numpy as np
import pandas as pd

DRIVE_ROOT = Path("/content/drive/MyDrive")
PFI_ROOT = DRIVE_ROOT / "PFI_MVP"

# Checkpoints finales del producto
SAGITTAL_CKPT = PFI_ROOT / "models/final/sagittal_spider_multiclass_final_best.pt"
AXIAL_CKPT    = PFI_ROOT / "models/final/axial_t2_alkafri_final_best.pt"

# Salidas QUAL
QUAL_ROOT = PFI_ROOT / "qual"
HELDOUT_ROOT = QUAL_ROOT / "heldout"
REPORTS_ROOT = QUAL_ROOT / "reports"
DOCS_ROOT = PFI_ROOT / "docs"

SAG_TEST_DIR = HELDOUT_ROOT / "sagittal"
AX_TEST_DIR  = HELDOUT_ROOT / "axial"

SAG_IMAGES_OUT = SAG_TEST_DIR / "images"
SAG_MASKS_OUT  = SAG_TEST_DIR / "masks"
AX_IMAGES_OUT  = AX_TEST_DIR / "images"
AX_MASKS_OUT   = AX_TEST_DIR / "masks"

for d in [REPORTS_ROOT, DOCS_ROOT, SAG_IMAGES_OUT, SAG_MASKS_OUT, AX_IMAGES_OUT, AX_MASKS_OUT]:
    d.mkdir(parents=True, exist_ok=True)

# Si True, limpia el held-out creado anteriormente antes de copiar.
CLEAR_EXISTING_HELDOUT = True

# Tamaño esperado por los modelos finales
TARGET_SIZE = 256

print("PFI_ROOT:", PFI_ROOT, "| exists:", PFI_ROOT.exists())
print("SAGITTAL_CKPT:", SAGITTAL_CKPT, "| exists:", SAGITTAL_CKPT.exists())
print("AXIAL_CKPT:", AXIAL_CKPT, "| exists:", AXIAL_CKPT.exists())
print("SAG_TEST_DIR:", SAG_TEST_DIR)
print("AX_TEST_DIR:", AX_TEST_DIR)

PFI_ROOT: /content/drive/MyDrive/PFI_MVP | exists: True
SAGITTAL_CKPT: /content/drive/MyDrive/PFI_MVP/models/final/sagittal_spider_multiclass_final_best.pt | exists: True
AXIAL_CKPT: /content/drive/MyDrive/PFI_MVP/models/final/axial_t2_alkafri_final_best.pt | exists: True
SAG_TEST_DIR: /content/drive/MyDrive/PFI_MVP/qual/heldout/sagittal
AX_TEST_DIR: /content/drive/MyDrive/PFI_MVP/qual/heldout/axial


In [4]:
# ============================================
# 2) Utilidades comunes
# ============================================
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk_size)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def reset_dir(path: Path):
    if path.exists() and CLEAR_EXISTING_HELDOUT:
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

def copy_pair(image_src: Path, mask_src: Path, image_dst: Path, mask_dst: Path):
    image_dst.parent.mkdir(parents=True, exist_ok=True)
    mask_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(image_src, image_dst)
    shutil.copy2(mask_src, mask_dst)

def safe_rel(path: Path):
    try:
        return str(path.relative_to(PFI_ROOT))
    except Exception:
        return str(path)

def print_tree_counts():
    print("Sagital images:", len(list(SAG_IMAGES_OUT.glob("*"))))
    print("Sagital masks:", len(list(SAG_MASKS_OUT.glob("*"))))
    print("Axial images:", len(list(AX_IMAGES_OUT.glob("*"))))
    print("Axial masks:", len(list(AX_MASKS_OUT.glob("*"))))

## 3) Buscar splits/inventarios existentes

Esta celda intenta encontrar archivos que indiquen split, inventario, overview, train/validation/test, etc.

Revisá la salida. Si aparece un split limpio por paciente, conviene usarlo. Si no aparece, este notebook arma un held-out **preliminar fijo** con pares reales imagen/máscara.

In [5]:
# ============================================
# 3) Buscar archivos candidatos de split / inventario
# ============================================
patterns = [
    "*split*", "*overview*", "*holdout*", "*test*", "*train*", "*validation*", "*val*",
    "*pairing*", "*inventory*", "*manifest*"
]

valid_suffixes = {".csv", ".json", ".txt", ".parquet", ".tsv"}

hits = []
for pat in patterns:
    for p in PFI_ROOT.rglob(pat):
        if p.is_file() and p.suffix.lower() in valid_suffixes:
            if "qual/heldout" in str(p).replace("\\", "/").lower():
                continue
            hits.append(p)

hits = sorted(set(hits), key=lambda p: str(p).lower())

print("Archivos candidatos de split/inventario:", len(hits))
for i, p in enumerate(hits[:250]):
    print(f"{i:03d} | {p}")

split_scan_path = REPORTS_ROOT / "qual-002-split-candidates.txt"
split_scan_path.write_text("\n".join(str(p) for p in hits), encoding="utf-8")
print("\nGuardado:", split_scan_path)

Archivos candidatos de split/inventario: 83
000 | /content/drive/MyDrive/PFI_MVP/data/SPIDER/overview.csv
001 | /content/drive/MyDrive/PFI_MVP/results/E10_axial_t2_final_training_clean/E10_axial_t2_final_training_clean_report.json
002 | /content/drive/MyDrive/PFI_MVP/results/E10_axial_t2_final_training_clean/E10_test_metrics.json
003 | /content/drive/MyDrive/PFI_MVP/results/E10_axial_t2_final_training_clean/E10_training_history.csv
004 | /content/drive/MyDrive/PFI_MVP/results/E10_axial_t2_final_training_clean/E10_val_metrics.json
005 | /content/drive/MyDrive/PFI_MVP/results/E11_axial_class_mapping_final_report/E11_axial_class_distribution_by_split.csv
006 | /content/drive/MyDrive/PFI_MVP/results/E12_sagittal_final_training_clean/E12_json_metrics_inventory.csv
007 | /content/drive/MyDrive/PFI_MVP/results/E15_backend_mvp_translation/E15_service_file_inventory.csv
008 | /content/drive/MyDrive/PFI_MVP/results/E15_backend_mvp_translation/E15_smoke_test_report.json
009 | /content/drive/MyDri

In [6]:
# ============================================
# 3b) Preview automático de CSV/TSV/JSON chicos
# ============================================
def preview_table_file(path: Path, max_rows=5):
    print("\n" + "=" * 100)
    print(path)
    try:
        if path.suffix.lower() in [".csv", ".tsv"]:
            sep = "\t" if path.suffix.lower() == ".tsv" else ","
            df = pd.read_csv(path, sep=sep)
            print("shape:", df.shape)
            print("columns:", list(df.columns)[:30])
            for col in df.columns:
                lc = col.lower()
                if lc in ["split", "subset", "fold", "set", "phase", "partition"]:
                    print(f"value_counts[{col}]:")
                    print(df[col].value_counts(dropna=False).head(20))
            print(df.head(max_rows).to_string(index=False))
        elif path.suffix.lower() == ".json":
            text = path.read_text(encoding="utf-8", errors="ignore")
            obj = json.loads(text)
            if isinstance(obj, dict):
                print("json keys:", list(obj.keys())[:30])
            elif isinstance(obj, list):
                print("json list len:", len(obj))
                print("first item:", obj[0] if obj else None)
    except Exception as e:
        print("No pude previsualizar:", repr(e))

# Para no saturar Colab, preview de los primeros 25.
for p in hits[:25]:
    preview_table_file(p)


/content/drive/MyDrive/PFI_MVP/data/SPIDER/overview.csv
shape: (447, 39)
columns: ['new_file_name', 'num_vertebrae', 'num_discs', 'sex', 'birth_date', 'subset', 'AngioFlag', 'BodyPartExamined', 'DeviceSerialNumber', 'EchoNumbers', 'EchoTime', 'EchoTrainLength', 'FlipAngle', 'ImagedNucleus', 'ImagingFrequency', 'InPlanePhaseEncodingDirection', 'MRAcquisitionType', 'MagneticFieldStrength', 'Manufacturer', 'ManufacturerModelName', 'NumberOfPhaseEncodingSteps', 'PercentPhaseFieldOfView', 'PercentSampling', 'PhotometricInterpretation', 'PixelBandwidth', 'PixelSpacing', 'RepetitionTime', 'SAR', 'SamplesPerPixel', 'ScanningSequence']
value_counts[subset]:
subset
training      360
validation     87
Name: count, dtype: int64
new_file_name  num_vertebrae  num_discs sex  birth_date   subset AngioFlag BodyPartExamined  DeviceSerialNumber  EchoNumbers  EchoTime  EchoTrainLength  FlipAngle ImagedNucleus  ImagingFrequency InPlanePhaseEncodingDirection MRAcquisitionType  MagneticFieldStrength        

## 4) Verificar pares sagitales SPIDER

Busca pares `image/mask` reales. Por defecto toma archivos de:

- `data/SPIDER/images/images`
- `data/SPIDER/masks/masks`

El notebook prioriza nombres exactos, por ejemplo `101_t2.mha`.

In [7]:
# ============================================
# 4) Pares sagitales SPIDER
# ============================================
SAG_IMAGES_SRC = PFI_ROOT / "data/SPIDER/images/images"
SAG_MASKS_SRC = PFI_ROOT / "data/SPIDER/masks/masks"

print("SAG_IMAGES_SRC:", SAG_IMAGES_SRC, "| exists:", SAG_IMAGES_SRC.exists())
print("SAG_MASKS_SRC:", SAG_MASKS_SRC, "| exists:", SAG_MASKS_SRC.exists())

def find_sag_mask_for_image(img: Path):
    # 1) mismo nombre
    exact = SAG_MASKS_SRC / img.name
    if exact.exists():
        return exact

    # 2) mismo stem con extensiones compatibles
    candidates = list(SAG_MASKS_SRC.glob(img.stem + "*"))
    candidates = [p for p in candidates if p.is_file()]
    if candidates:
        return sorted(candidates, key=lambda p: str(p))[0]

    # 3) por id inicial antes de _t2
    case_id = img.name.split("_")[0]
    candidates = list(SAG_MASKS_SRC.glob(case_id + "*"))
    candidates = [p for p in candidates if p.is_file()]
    if candidates:
        return sorted(candidates, key=lambda p: str(p))[0]

    return None

sag_pairs = []
missing_sag_masks = []

for img in sorted(SAG_IMAGES_SRC.glob("*.mha")):
    # Evitar SPACE para una primera corrida simple, salvo que quieras incluirlos explícitamente.
    if "_SPACE" in img.name:
        continue
    mask = find_sag_mask_for_image(img)
    if mask is not None and mask.exists():
        sag_pairs.append((img, mask))
    else:
        missing_sag_masks.append(img)

print("Pares sagitales encontrados:", len(sag_pairs))
print("Imágenes sagitales sin máscara detectada:", len(missing_sag_masks))

for img, mask in sag_pairs[:20]:
    print(img.name, "|", mask.name)

SAG_IMAGES_SRC: /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images | exists: True
SAG_MASKS_SRC: /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks | exists: True
Pares sagitales encontrados: 406
Imágenes sagitales sin máscara detectada: 0
100_t1.mha | 100_t1.mha
100_t2.mha | 100_t2.mha
101_t1.mha | 101_t1.mha
101_t2.mha | 101_t2.mha
104_t1.mha | 104_t1.mha
104_t2.mha | 104_t2.mha
105_t1.mha | 105_t1.mha
105_t2.mha | 105_t2.mha
106_t1.mha | 106_t1.mha
106_t2.mha | 106_t2.mha
107_t1.mha | 107_t1.mha
107_t2.mha | 107_t2.mha
108_t1.mha | 108_t1.mha
108_t2.mha | 108_t2.mha
109_t1.mha | 109_t1.mha
109_t2.mha | 109_t2.mha
10_t1.mha | 10_t1.mha
10_t2.mha | 10_t2.mha
110_t1.mha | 110_t1.mha
110_t2.mha | 110_t2.mha


## 5) Verificar pares axiales Al-Kafri/Sudirman

Busca pares procesados:

- `pair_XXXX_image.npy`
- `pair_XXXX_mask.npy`

en `data/AXIAL_ALKAFRI/processed/pairing_v1`.

In [8]:
# ============================================
# 5) Pares axiales procesados
# ============================================
AXIAL_PAIRING_SRC = PFI_ROOT / "data/AXIAL_ALKAFRI/processed/pairing_v1"

print("AXIAL_PAIRING_SRC:", AXIAL_PAIRING_SRC, "| exists:", AXIAL_PAIRING_SRC.exists())

axial_pairs = []
for img in sorted(AXIAL_PAIRING_SRC.glob("*_image.npy")):
    mask = AXIAL_PAIRING_SRC / img.name.replace("_image.npy", "_mask.npy")
    if mask.exists():
        pair_id = img.name.replace("_image.npy", "")
        axial_pairs.append((pair_id, img, mask))

print("Pares axiales encontrados:", len(axial_pairs))
for pair_id, img, mask in axial_pairs[:20]:
    print(pair_id, "|", img.name, "|", mask.name)

# Chequeo rápido de shape/rango del primer axial
if axial_pairs:
    arr = np.load(axial_pairs[0][1])
    msk = np.load(axial_pairs[0][2])
    print("\nPrimer axial image shape/dtype/range:", arr.shape, arr.dtype, float(np.nanmin(arr)), float(np.nanmax(arr)))
    print("Primer axial mask shape/dtype/range:", msk.shape, msk.dtype, float(np.nanmin(msk)), float(np.nanmax(msk)))

AXIAL_PAIRING_SRC: /content/drive/MyDrive/PFI_MVP/data/AXIAL_ALKAFRI/processed/pairing_v1 | exists: True
Pares axiales encontrados: 718
pair_0001 | pair_0001_image.npy | pair_0001_mask.npy
pair_0002 | pair_0002_image.npy | pair_0002_mask.npy
pair_0003 | pair_0003_image.npy | pair_0003_mask.npy
pair_0004 | pair_0004_image.npy | pair_0004_mask.npy
pair_0005 | pair_0005_image.npy | pair_0005_mask.npy
pair_0006 | pair_0006_image.npy | pair_0006_mask.npy
pair_0007 | pair_0007_image.npy | pair_0007_mask.npy
pair_0008 | pair_0008_image.npy | pair_0008_mask.npy
pair_0009 | pair_0009_image.npy | pair_0009_mask.npy
pair_0010 | pair_0010_image.npy | pair_0010_mask.npy
pair_0011 | pair_0011_image.npy | pair_0011_mask.npy
pair_0012 | pair_0012_image.npy | pair_0012_mask.npy
pair_0013 | pair_0013_image.npy | pair_0013_mask.npy
pair_0014 | pair_0014_image.npy | pair_0014_mask.npy
pair_0015 | pair_0015_image.npy | pair_0015_mask.npy
pair_0016 | pair_0016_image.npy | pair_0016_mask.npy
pair_0017 | pair

## 6) Selección del held-out

Por defecto este notebook usa una selección preliminar fija:

- Sagital: 10 casos SPIDER.
- Axial: 30 pares Al-Kafri procesados.

Podés reemplazar las listas por IDs de un split limpio si encontraste un CSV/JSON confiable.

Si no confirmás que esos casos nunca estuvieron en train, el manifest queda marcado como:

`preliminary_gt_not_confirmed_unseen`

In [9]:
# ============================================
# 6) Selección del held-out
# ============================================

# Cambiar a "clean_test_from_split" si tenés un split limpio confirmado.
SELECTION_MODE = "preliminary_gt_not_confirmed_unseen"

# Opcional: reemplazar con IDs limpios de split si los tenés.
# Sagital: nombres de archivo .mha, por ejemplo ["101_t2.mha", "104_t2.mha"]
USER_SAG_CASE_NAMES = None

# Axial: ids tipo pair_0001, pair_0002...
USER_AXIAL_PAIR_IDS = None

DEFAULT_SAG_CASE_NAMES = [
    "101_t2.mha",
    "104_t2.mha",
    "105_t2.mha",
    "106_t2.mha",
    "107_t2.mha",
    "109_t2.mha",
    "113_t2.mha",
    "115_t2.mha",
    "116_t2.mha",
    "118_t2.mha",
]

DEFAULT_AXIAL_PAIR_IDS = [f"pair_{i:04d}" for i in range(1, 31)]

sag_case_names = USER_SAG_CASE_NAMES or DEFAULT_SAG_CASE_NAMES
axial_pair_ids = USER_AXIAL_PAIR_IDS or DEFAULT_AXIAL_PAIR_IDS

available_sag_by_name = {img.name: (img, mask) for img, mask in sag_pairs}
selected_sag = []
for name in sag_case_names:
    if name in available_sag_by_name:
        selected_sag.append((name, *available_sag_by_name[name]))

if len(selected_sag) < len(sag_case_names):
    already = {name for name, _, _ in selected_sag}
    for img, mask in sag_pairs:
        if img.name not in already:
            selected_sag.append((img.name, img, mask))
        if len(selected_sag) >= len(sag_case_names):
            break

available_ax_by_id = {pair_id: (img, mask) for pair_id, img, mask in axial_pairs}
selected_axial = []
for pair_id in axial_pair_ids:
    if pair_id in available_ax_by_id:
        selected_axial.append((pair_id, *available_ax_by_id[pair_id]))

if len(selected_axial) < len(axial_pair_ids):
    already = {pair_id for pair_id, _, _ in selected_axial}
    for pair_id, img, mask in axial_pairs:
        if pair_id not in already:
            selected_axial.append((pair_id, img, mask))
        if len(selected_axial) >= len(axial_pair_ids):
            break

print("SELECTION_MODE:", SELECTION_MODE)
print("Selected sagittal:", len(selected_sag))
for item in selected_sag:
    print(" -", item[0], "|", item[1], "|", item[2])

print("\nSelected axial:", len(selected_axial))
for item in selected_axial[:40]:
    print(" -", item[0], "|", item[1].name, "|", item[2].name)

if len(selected_sag) == 0:
    raise RuntimeError("No se seleccionó ningún caso sagital. Revisar rutas/máscaras SPIDER.")
if len(selected_axial) == 0:
    raise RuntimeError("No se seleccionó ningún caso axial. Revisar pairing_v1.")

SELECTION_MODE: preliminary_gt_not_confirmed_unseen
Selected sagittal: 10
 - 101_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/101_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/101_t2.mha
 - 104_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/104_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/104_t2.mha
 - 105_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/105_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/105_t2.mha
 - 106_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/106_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/106_t2.mha
 - 107_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/107_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/107_t2.mha
 - 109_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/images/images/109_t2.mha | /content/drive/MyDrive/PFI_MVP/data/SPIDER/masks/masks/109_t2.mha
 - 113_t2.mha 

In [10]:
# ============================================
# 7) Crear carpetas held-out images/ + masks/
# ============================================
reset_dir(SAG_IMAGES_OUT)
reset_dir(SAG_MASKS_OUT)
reset_dir(AX_IMAGES_OUT)
reset_dir(AX_MASKS_OUT)

manifest_rows = []

# Sagital: mantener mismo nombre en images y masks.
for case_id, img, mask in selected_sag:
    img_dst = SAG_IMAGES_OUT / img.name
    mask_dst = SAG_MASKS_OUT / mask.name
    copy_pair(img, mask, img_dst, mask_dst)

    manifest_rows.append({
        "plane": "sagittal",
        "case_id": case_id,
        "image_source": str(img),
        "mask_source": str(mask),
        "image_dest": str(img_dst),
        "mask_dest": str(mask_dst),
        "selection_mode": SELECTION_MODE,
    })

# Axial: renombrar a pair_XXXX.npy en ambas carpetas para matching simple.
for pair_id, img, mask in selected_axial:
    img_dst = AX_IMAGES_OUT / f"{pair_id}.npy"
    mask_dst = AX_MASKS_OUT / f"{pair_id}.npy"
    copy_pair(img, mask, img_dst, mask_dst)

    manifest_rows.append({
        "plane": "axial",
        "case_id": pair_id,
        "image_source": str(img),
        "mask_source": str(mask),
        "image_dest": str(img_dst),
        "mask_dest": str(mask_dst),
        "selection_mode": SELECTION_MODE,
    })

print_tree_counts()

manifest_df = pd.DataFrame(manifest_rows)

manifest_csv = REPORTS_ROOT / "qual-002-heldout-manifest.csv"
manifest_json = REPORTS_ROOT / "qual-002-heldout-manifest.json"

manifest_df.to_csv(manifest_csv, index=False)
manifest_json.write_text(json.dumps({
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "selection_mode": SELECTION_MODE,
    "warning": (
        "Si selection_mode != clean_test_from_split, esta evaluacion debe reportarse como "
        "preliminar con ground truth, no como test limpio final."
    ),
    "sagittal_test_dir": str(SAG_TEST_DIR),
    "axial_test_dir": str(AX_TEST_DIR),
    "counts": {
        "sagittal": len(selected_sag),
        "axial": len(selected_axial),
    },
    "rows": manifest_rows,
}, indent=2, ensure_ascii=False), encoding="utf-8")

print("Manifest CSV:", manifest_csv)
print("Manifest JSON:", manifest_json)
manifest_df.head()

OSError: [Errno 107] Transport endpoint is not connected

## 8) Ubicar repo y `evaluate_model.py`

Este notebook intenta ubicar automáticamente el repo donde está `scripts/evaluate_model.py`.

Si no lo encuentra, cloná o montá el repo AI Module / monorepo en Drive y re-ejecutá esta celda.

In [ ]:
# ============================================
# 8) Ubicar repo y evaluator
# ============================================
REPO_CANDIDATES = [
    PFI_ROOT / "repo",
    PFI_ROOT / "PFI_MVPTest_Enzo_AImodule",
    DRIVE_ROOT / "PFI_MVPTest_Enzo_AImodule",
    Path("/content/PFI_MVPTest_Enzo_AImodule"),
    Path.cwd(),
]

found = []
for repo in REPO_CANDIDATES:
    script = repo / "scripts/evaluate_model.py"
    if script.exists():
        found.append((repo, script))

if not found:
    for script in PFI_ROOT.rglob("evaluate_model.py"):
        if script.name == "evaluate_model.py":
            repo = script.parent.parent
            found.append((repo, script))

print("Evaluators encontrados:", len(found))
for i, (repo, script) in enumerate(found):
    print(f"{i:02d} | repo={repo} | script={script}")

if not found:
    raise RuntimeError(
        "No encontré scripts/evaluate_model.py. "
        "Subí/cloná el repo en /content/drive/MyDrive/PFI_MVP/repo o ajustá REPO_CANDIDATES."
    )

REPO_ROOT, EVALUATE_SCRIPT = found[0]
print("\nUsando REPO_ROOT:", REPO_ROOT)
print("Usando EVALUATE_SCRIPT:", EVALUATE_SCRIPT)

pythonpath_candidates = []
if (REPO_ROOT / "ai_service").exists():
    pythonpath_candidates.append(REPO_ROOT / "ai_service")
pythonpath_candidates.append(REPO_ROOT)

PYTHONPATH_VALUE = str(pythonpath_candidates[0])
os.environ["PYTHONPATH"] = PYTHONPATH_VALUE

print("PYTHONPATH:", os.environ["PYTHONPATH"])

## 9) Ejecutar QUAL-004

Corre el evaluador para ambos planos y guarda:

- `docs/qual-004-sagittal.json`
- `docs/qual-004-axial.json`
- `docs/qual-004-sagittal.log`
- `docs/qual-004-axial.log`

In [ ]:
# ============================================
# 9) Ejecutar evaluate_model.py
# ============================================
def run_eval(plane: str, checkpoint: Path, test_dir: Path, num_classes: int, output_json: Path):
    log_path = output_json.with_suffix(".log")

    cmd = [
        sys.executable,
        str(EVALUATE_SCRIPT),
        "--plane", plane,
        "--checkpoint", str(checkpoint),
        "--test-dir", str(test_dir),
        "--num-classes", str(num_classes),
        "--target-size", str(TARGET_SIZE),
        "--output", str(output_json),
    ]

    env = os.environ.copy()
    env["PYTHONPATH"] = os.environ.get("PYTHONPATH", "")

    print("\n" + "=" * 100)
    print("Running:", " ".join(cmd))
    print("cwd:", REPO_ROOT)
    print("PYTHONPATH:", env["PYTHONPATH"])

    proc = subprocess.run(
        cmd,
        cwd=str(REPO_ROOT),
        env=env,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )

    log_text = proc.stdout
    log_path.write_text(log_text, encoding="utf-8", errors="ignore")

    print(log_text[-5000:])
    print("returncode:", proc.returncode)
    print("log:", log_path)
    print("output exists:", output_json.exists(), output_json)

    if proc.returncode != 0:
        raise RuntimeError(f"Falló evaluación {plane}. Ver log: {log_path}")

    return output_json, log_path

sag_out_json = DOCS_ROOT / "qual-004-sagittal.json"
ax_out_json  = DOCS_ROOT / "qual-004-axial.json"

sag_json, sag_log = run_eval(
    plane="sagittal",
    checkpoint=SAGITTAL_CKPT,
    test_dir=SAG_TEST_DIR,
    num_classes=4,
    output_json=sag_out_json,
)

ax_json, ax_log = run_eval(
    plane="axial",
    checkpoint=AXIAL_CKPT,
    test_dir=AX_TEST_DIR,
    num_classes=6,
    output_json=ax_out_json,
)

print("\nEvaluación finalizada.")
print("Sagital:", sag_json)
print("Axial:", ax_json)

## 10) Leer JSONs y generar resumen para Claude/Codex

In [ ]:
# ============================================
# 10) Mostrar resultados y generar resumen
# ============================================
def load_json(path: Path):
    if not path.exists():
        print("NO EXISTE:", path)
        return None
    return json.loads(path.read_text(encoding="utf-8"))

def find_metric_recursive(obj, possible_keys):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in possible_keys:
                return v
        for v in obj.values():
            found = find_metric_recursive(v, possible_keys)
            if found is not None:
                return found
    elif isinstance(obj, list):
        for item in obj:
            found = find_metric_recursive(item, possible_keys)
            if found is not None:
                return found
    return None

def pretty_json(obj):
    return json.dumps(obj, indent=2, ensure_ascii=False, default=str)

sag_result = load_json(sag_out_json)
ax_result = load_json(ax_out_json)

print("\n==== qual-004-sagittal.json ====")
print(pretty_json(sag_result)[:12000] if sag_result is not None else "NO JSON")

print("\n==== qual-004-axial.json ====")
print(pretty_json(ax_result)[:12000] if ax_result is not None else "NO JSON")

sag_macro = find_metric_recursive(sag_result, {
    "diceMacroForeground", "dice_macro_foreground", "macro_dice_foreground",
    "diceMacroNoBackground", "dice_macro_no_bg"
}) if sag_result is not None else None

ax_macro = find_metric_recursive(ax_result, {
    "diceMacroForeground", "dice_macro_foreground", "macro_dice_foreground",
    "diceMacroNoBackground", "dice_macro_no_bg"
}) if ax_result is not None else None

summary_md = f'''
# QUAL-004 - Evaluación preliminar con Ground Truth

Fecha UTC: {datetime.now(timezone.utc).isoformat()}

## Advertencia metodológica

selection_mode: `{SELECTION_MODE}`

Si `selection_mode` no es `clean_test_from_split`, estos números deben reportarse como
**evaluación preliminar con ground truth**, no como test limpio final por paciente.

## Inputs

Sagital:
- test_dir: `{SAG_TEST_DIR}`
- checkpoint: `{SAGITTAL_CKPT}`
- casos: {len(selected_sag)}

Axial:
- test_dir: `{AX_TEST_DIR}`
- checkpoint: `{AXIAL_CKPT}`
- casos: {len(selected_axial)}

## Outputs

- `{sag_out_json}`
- `{ax_out_json}`
- `{sag_log}`
- `{ax_log}`
- `{manifest_json}`

## Métricas principales detectadas

- Sagital dice macro foreground/no-bg: `{sag_macro}`
- Axial dice macro foreground/no-bg: `{ax_macro}`

## Nota para Claude/Codex

Los JSON completos están guardados en `docs/qual-004-sagittal.json` y
`docs/qual-004-axial.json`. Usar esos archivos como fuente de verdad para
QUAL-005 / planificación de mejora de entrenamiento.
'''.strip()

summary_path = DOCS_ROOT / "qual-004-summary.md"
summary_path.write_text(summary_md, encoding="utf-8")

print("\n==== SUMMARY PARA CLAUDE/CODEX ====")
print(summary_md)
print("\nGuardado:", summary_path)

## 11) Comprimir evidencia QUAL

Genera un zip con manifest, JSONs, logs y summary para descargar o compartir.

In [ ]:
# ============================================
# 11) Zip de evidencia QUAL
# ============================================
import zipfile

zip_path = REPORTS_ROOT / "qual-002-004-evidence.zip"

files_to_zip = [
    REPORTS_ROOT / "qual-002-split-candidates.txt",
    REPORTS_ROOT / "qual-002-heldout-manifest.csv",
    REPORTS_ROOT / "qual-002-heldout-manifest.json",
    DOCS_ROOT / "qual-004-sagittal.json",
    DOCS_ROOT / "qual-004-axial.json",
    DOCS_ROOT / "qual-004-sagittal.log",
    DOCS_ROOT / "qual-004-axial.log",
    DOCS_ROOT / "qual-004-summary.md",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in files_to_zip:
        if p.exists():
            z.write(p, arcname=p.name)
        else:
            print("No existe, no se agrega:", p)

print("ZIP:", zip_path)
print("ZIP exists:", zip_path.exists())
if zip_path.exists():
    print("ZIP size MB:", round(zip_path.stat().st_size / (1024 * 1024), 3))
    print("ZIP sha256:", sha256_file(zip_path))

## 12) Mensaje sugerido para Claude/Codex

Cuando termine la ejecución, pegá este bloque junto con el contenido de `qual-004-summary.md` o los JSON completos.

In [ ]:
print(f'''
QUAL-002 / QUAL-004 ejecutado en Colab.

Held-out:
- selection_mode: {SELECTION_MODE}
- sagital casos: {len(selected_sag)}
- axial casos: {len(selected_axial)}

Carpetas:
- sagital test_dir: {SAG_TEST_DIR}
- axial test_dir: {AX_TEST_DIR}

Outputs:
- {DOCS_ROOT / "qual-004-sagittal.json"}
- {DOCS_ROOT / "qual-004-axial.json"}
- {DOCS_ROOT / "qual-004-summary.md"}
- {REPORTS_ROOT / "qual-002-heldout-manifest.json"}

Aclaración:
Si selection_mode != clean_test_from_split, reportar como evaluación preliminar con GT,
no como test limpio final por paciente.

Copio a continuación el summary y/o los dos JSON.
''')